# 🚩 intraPSIC
## `model_personae.ipynb`

> `model_personae.ipynb`<br>
> Simone J. Skeen x Claude Code (08-05-2026)<br>
> WIP - NOT FOR DISTRIBUTION

In [ ]:
%%capture
# === INSTALL DEPENDENCIES === #

%pip install -r ../requirements.txt

In [ ]:
# === STANDARD LIBRARY IMPORTS === #

import sys
import warnings
from pathlib import Path

# === THIRD-PARTY IMPORTS === #

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# IPython display configuration
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# === PANDAS DISPLAY OPTIONS === #
# Show all columns and rows for debugging.

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# === SUPPRESS WARNINGS === #
# Hide FutureWarning and UserWarning to keep output clean.

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

# === LOAD ENVIRONMENT VARIABLES === #
# The .env file should contain KNOW_DIR and optionally OLLAMA_HOST.

load_dotenv(Path('..') / '.env')

In [ ]:
# Import labeled dissertation data

d_ = pd.read_csv('d_inf_labeled_long.csv')

In [ ]:
# Drop `text` column

# Condense to classifier confidence >0.90

# Rewrite to Parquet

In [ ]:
%pip install stepmix

In [ ]:
import numpy as np
import pandas as pd

from stepmix.stepmix import StepMix

In [ ]:
# Simulate

N = 10_000
cols = [f'v{i:02d}' for i in range(10)]

d = pd.DataFrame(
    np.random.randint(0, 2, size=(N, len(cols))),
    columns=cols
)

# Inspect & verify

d

In [ ]:
# Latent class analysis / StepMix intro

### NOTE 8/5: working from the docs, this cell

# Categorical StepMix Model with 3 latent classes
model = StepMix(
    n_components=3, 
    measurement="binary", 
    verbose=1, 
    max_iter=5000, 
    n_init=20, 
    random_state=56,
    )
model.fit(d)

# Allow missing values
#model_nan = StepMix(n_components=3, measurement="binary_nan", random_state=56)
#model_nan.fit(d)

### NOTE: StepMix supports LatentGOLD-equivalent coviariates of class membership + distal outcomes

### NOTE: excellent model selection tutorial: https://colab.research.google.com/drive/1btXHCx90eCsnUlQv_yN-9AzKDhJP_JkG?usp=drive_link


In [ ]:
%%capture
# Grid search: n_components (1-8)

from tqdm import tqdm

param_grid = {
    'n_components': range(1, 9)
}

results = []

for n_comp in tqdm(param_grid['n_components'], desc='n_components'):
    model = StepMix(
        n_components=n_comp,
        measurement='binary',
        max_iter=5000,
        n_init=20,
        random_state=56,
        verbose=0
    )
    model.fit(d)
    
    results.append({
        'n_components': n_comp,
        'LL': model.score(d) * len(d),
        'AIC': model.aic(d),
        'BIC': model.bic(d),
        'CAIC': model.caic(d),
        'SABIC': model.sabic(d),
        'entropy': model.relative_entropy(d)
    })

In [ ]:

# Inspect results
results_df = pd.DataFrame(results)
results_df.sort_values('BIC')